In [1]:
import os
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

from sdlarch_rl import make
from sdlarch_rl.utils.discretizer import MainDiscretizer

from sdlarch_rl.utils.utils import (
    get_latest_model,
    FrameSkip,
    TimeLimit,
    ExcludeButtonsWrapper,
    AugmentObservation,
    RandomStateWrapper,
)

from stable_baselines3.common.atari_wrappers import WarpFrame
from tianshou.env import SubprocVectorEnv, DummyVectorEnv

# from tianshou.data import Collector, VectorReplayBuffer
from tianshou.policy import PPOPolicy
# from tianshou.trainer import OnpolicyTrainer

# from tianshou.env import AsyncVectorEnv
from tianshou.data import AsyncCollector, VectorReplayBuffer
from tianshou.trainer import OffpolicyTrainer

from torch.utils.tensorboard import SummaryWriter
from gym.wrappers import FrameStack

# =====================================================
# Configs
# =====================================================

NUM_ENV = 4
SAVE_DIR = Path("./model-tianshou-sf4")
TENSORBOARD = "./tensorboard-tianshousf4"
TOTAL_TIMESTEP_NUMB = 500_000_000
MAX_STEPS = 4000
CHECK_FREQ_NUMB = 5000

ENT_COEF = 0.001
n_steps = 2048
batch_size = 64 * NUM_ENV

writer = SummaryWriter(TENSORBOARD)
SAVE_DIR.mkdir(exist_ok=True)

global_step = 0

# =====================================================
# Create Env (same as SB3)
# =====================================================

def make_env():
    def _init():
        env = make("SuperStreetFighterIV-3DS")
        env = RandomStateWrapper(env, states=[
            'default',
            'ryu_ken_easy_south_asia',
            'ryu_guile_easy_skyscraper',
            'ryu_chunli_easy_asia_south'
        ])

        buttons = env.unwrapped.buttons
        to_exclude = ["START", "SELECT", "L2", "R2", "L3", "R3", "A", "X", "Y"]

        env = ExcludeButtonsWrapper(env, buttons, to_exclude)
        env = AugmentObservation(env)
        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=6, stochastic=True)
        env = TimeLimit(env, max_steps=MAX_STEPS)
        env = FrameStack(env, 4)

        return env
    return _init


# =====================================================
# VecEnv
# =====================================================

if NUM_ENV == 1:
    venv = DummyVectorEnv([make_env()])
else:
    venv = SubprocVectorEnv([make_env() for _ in range(NUM_ENV)])

test_env = DummyVectorEnv([make_env()])

# venv = AsyncVectorEnv([make_env() for _ in range(NUM_ENV)])
# test_env = AsyncVectorEnv([make_env()])


# -----------------------------
# Backbone + Actor + Critic
# -----------------------------
class CNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        # note: input shape expected N,H,W,C (NHWC) from your VecFrameStack wrapper
        # we'll permute to NCHW inside forward of actor/critic
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
            nn.Flatten()
        )
        # compute flattened feature dim
        with torch.no_grad():
            dummy = torch.zeros(1, 4, 96, 96)  # NCHW for conv
            feat = self.conv(dummy)
            self.feat_dim = feat.shape[1]

    def forward(self, obs_nchw):
        # obs_nchw expected NCHW already
        return self.conv(obs_nchw)


class ActorNet(nn.Module):
    def __init__(self, backbone: CNNBackbone, action_dim: int):
        super().__init__()
        self.backbone = backbone
        self.logits = nn.Linear(self.backbone.feat_dim, action_dim)

    def forward(self, obs, state=None, info=None):
        # Tianshou passes obs as NHWC by default in your setup,
        # so convert to NCHW here (handles numpy/tensor shapes)
        if isinstance(obs, np.ndarray):
            obs_t = torch.as_tensor(obs).float().permute(0, 3, 1, 2)
        else:
            # torch.Tensor
            obs_t = obs.permute(0, 3, 1, 2)
        feat = self.backbone(obs_t)
        logits = self.logits(feat)  # shape [B, action_dim]
        return logits, state


class CriticNet(nn.Module):
    def __init__(self, backbone: CNNBackbone):
        super().__init__()
        self.backbone = backbone
        self.v = nn.Linear(self.backbone.feat_dim, 1)

    def forward(self, obs, state=None, info=None):
        if isinstance(obs, np.ndarray):
            obs_t = torch.as_tensor(obs).float().permute(0, 3, 1, 2)
        else:
            obs_t = obs.permute(0, 3, 1, 2)
        feat = self.backbone(obs_t)
        value = self.v(feat).squeeze(-1)  # shape [B]
        return value, state


# -----------------------------
# instantiate nets and policy
# -----------------------------
# detect action dim from action_space (assume Discrete or MultiBinary -> map to discrete count)
if isinstance(venv.action_space, list): 
    action_space = venv.action_space[0] 
else: 
    action_space = venv.action_space


if hasattr(action_space, "n"):
    action_dim = int(action_space.n)
elif hasattr(action_space, "shape"):
    # fallback for binary/multi-binary -> convert to number of discrete combos if you used discretizer
    # but here we just use action_space.n when discretizer applied. If not, convert externally.
    action_dim = int(np.prod(action_space.shape))
else:
    raise RuntimeError("Cannot infer action dim from action_space")

backbone = CNNBackbone()
actor = ActorNet(backbone, action_dim)
critic = CriticNet(backbone)

# optimizer should include both actor + critic params
optim = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=2.5e-4)

# now create PPOPolicy with actor and critic modules (these accept state/info)
policy = PPOPolicy(
    actor=actor,
    critic=critic,
    optim=optim,
    dist_fn=torch.distributions.Categorical,
    action_space=action_space,
    discount_factor=0.99,
    gae_lambda=0.95,
    max_grad_norm=0.5,
    vf_coef=0.5,
    ent_coef=ENT_COEF,
    eps_clip=0.1,
    advantage_normalization=True,
    recompute_advantage=True,
    action_scaling=False,
)

# =====================================================
# Collector
# =====================================================

buffer = VectorReplayBuffer(n_steps * NUM_ENV, NUM_ENV)

# train_collector = Collector(policy, venv, buffer, exploration_noise=True)
# test_collector = Collector(policy, test_env)

train_collector = AsyncCollector(
    policy,
    venv,
    buffer,
    exploration_noise=True,
)

test_collector = AsyncCollector(
    policy,
    test_env,
    VectorReplayBuffer(10000, 1),
    exploration_noise=False
)


# =====================================================
# Load previous checkpoint
# =====================================================

latest_model_path = get_latest_model(SAVE_DIR)

if latest_model_path:
    print(f"Loading checkpoint: {latest_model_path}")
    policy.load_state_dict(torch.load(latest_model_path))
else:
    print("No previous checkpoint found — starting fresh.")


# =====================================================
# Training Loop (Tianshou version)
# =====================================================

def save_best_fn(policy):
    save_path = SAVE_DIR / f"best_model_{global_step}.zip"
    torch.save(policy.state_dict(), save_path)
    print("Saved:", save_path)

def train_fn(epoch, env_step):
    global global_step
    global_step = env_step


# trainer = OnpolicyTrainer(
#     policy=policy,
#     train_collector=train_collector,
#     test_collector=test_collector,
#     max_epoch=int(TOTAL_TIMESTEP_NUMB / (n_steps * NUM_ENV)),
#     step_per_epoch=n_steps * NUM_ENV,
#     step_per_collect=n_steps,
#     repeat_per_collect=10,
#     episode_per_test=2,
#     batch_size=batch_size,
#     train_fn=train_fn,
#     save_best_fn=save_best_fn,
#     logger=writer,
# )

trainer = OffpolicyTrainer(
    policy=policy,
    buffer=buffer,
    train_collector=train_collector,
    test_collector=test_collector,
    max_epoch=int(TOTAL_TIMESTEP_NUMB / (n_steps * NUM_ENV)),
    step_per_epoch=n_steps * NUM_ENV,
    step_per_collect=0,         # collect continuoes
    update_per_step=1,          # PPO stable
    episode_per_test=2,
    batch_size=batch_size,
    train_fn=lambda epoch, env_step: buffer.clear(),   # keep on-policy
    save_best_fn=save_best_fn,
    logger=writer,
)

result = trainer.run()


# =====================================================
# Save final
# =====================================================

torch.save(policy.state_dict(), "final_sf4_policy.pth")
venv.close()


D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state


D:\Python311\Lib\site-packages\torch\_compile.py:31: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)
Please use Collector if not using async venv. Env class: SubprocVectorEnv
D:\Python311\Lib\site-packages\tianshou\data\collector.py:1138: UserWarning: Using async setting may collect extra transitions into buffer.
  warnings.warn("Using async setting may collect extra transitions into buffer.")
Please use Collector if not using async venv. Env class: DummyVectorEnv


No previous checkpoint found — starting fresh.


EOFError: 